In [6]:
import requests
from decimal import Decimal, getcontext

getcontext().prec = 28

def get_sol_market_data():
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "ids": "solana"
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    if not data:
        raise RuntimeError("CoinGecko returned empty response")

    row = data[0]

    price = Decimal(str(row["current_price"]))
    market_cap = Decimal(str(row["market_cap"]))
    circulating_supply = Decimal(str(row["circulating_supply"]))

    return {
        "name": row["name"],
        "symbol": row["symbol"],
        "price_usd": price,
        "market_cap_usd": market_cap,
        "circulating_supply": circulating_supply,
    }

def compute_from_mvrv(price, market_cap, circulating_supply, mvrv):
    mvrv = Decimal(str(mvrv))

    if mvrv <= 0:
        raise ValueError("MVRV must be > 0")

    realized_cap = market_cap / mvrv
    realized_price = price / mvrv
    discount_to_realized = (price / realized_price) - Decimal("1")

    return {
        "mvrv": mvrv,
        "realized_cap_usd": realized_cap,
        "realized_price_usd": realized_price,
        "discount_to_realized": discount_to_realized,
    }

if __name__ == "__main__":
    market = get_sol_market_data()

    # Вставь сюда MVRV из внешнего источника, например Glassnode
    # пример:
    mvrv_input = "0.65081472"

    result = compute_from_mvrv(
        price=market["price_usd"],
        market_cap=market["market_cap_usd"],
        circulating_supply=market["circulating_supply"],
        mvrv=mvrv_input,
    )

    print("Asset:", market["name"], f"({market['symbol'].upper()})")
    print("Price USD:", f"{market['price_usd']:.8f}")
    print("Market Cap USD:", f"{market['market_cap_usd']:.2f}")
    print("Circulating Supply:", f"{market['circulating_supply']:.8f}")
    print("MVRV:", f"{result['mvrv']:.8f}")
    print("Realized Cap USD:", f"{result['realized_cap_usd']:.2f}")
    print("Realized Price USD:", f"{result['realized_price_usd']:.8f}")
    print("Discount to Realized:", f"{result['discount_to_realized']:.2%}")

Asset: Solana (SOL)
Price USD: 84.42000000
Market Cap USD: 48426441006.00
Circulating Supply: 574204518.93391450
MVRV: 0.65081472
Realized Cap USD: 74408951607.61
Realized Price USD: 129.71433713
Discount to Realized: -34.92%


In [7]:
from dataclasses import dataclass, asdict


@dataclass
class MarketInputs:
    price: float
    realized_price: float
    stable_score: float        # 0..100
    dominance_score: float     # 0..100
    treasury_3m_yield: float   # например 3.678
    fed_rate: float            # например 3.75


@dataclass
class MarketResult:
    mvrv: float
    discount_to_realized_pct: float
    mvrv_score: float
    macro_score: float
    stable_score: float
    dominance_score: float
    final_score: float
    regime: str


def clamp(value: float, low: float = 0.0, high: float = 100.0) -> float:
    return max(low, min(high, value))


def compute_mvrv(price: float, realized_price: float) -> float:
    if realized_price <= 0:
        raise ValueError("realized_price must be > 0")
    return price / realized_price


def compute_discount_to_realized(price: float, realized_price: float) -> float:
    return (price / realized_price - 1.0) * 100.0


def score_mvrv(mvrv: float) -> float:
    """
    Чем ниже MVRV, тем выше score.
    Примерная шкала:
    <= 0.60  -> 95
    1.00     -> 80
    1.50     -> 50
    2.00     -> 20
    >= 2.50  -> 5
    """
    if mvrv <= 0.60:
        return 95.0
    elif mvrv <= 1.00:
        # 0.60..1.00 => 95..80
        return 95.0 - (mvrv - 0.60) / 0.40 * 15.0
    elif mvrv <= 1.50:
        # 1.00..1.50 => 80..50
        return 80.0 - (mvrv - 1.00) / 0.50 * 30.0
    elif mvrv <= 2.00:
        # 1.50..2.00 => 50..20
        return 50.0 - (mvrv - 1.50) / 0.50 * 30.0
    elif mvrv <= 2.50:
        # 2.00..2.50 => 20..5
        return 20.0 - (mvrv - 2.00) / 0.50 * 15.0
    else:
        return 5.0


def score_macro(treasury_3m_yield: float, fed_rate: float) -> float:
    """
    Логика:
    - чем ниже ставка и 3M yield, тем лучше для риска
    - чем меньше спред между ставкой и 3M, тем менее "строгий" кэш-режим
    """
    # Базовый score от уровня доходности 3M
    # 2% -> отлично, 5.5% -> плохо
    yield_score = 100.0 - ((treasury_3m_yield - 2.0) / (5.5 - 2.0)) * 100.0
    yield_score = clamp(yield_score)

    # Базовый score от ставки ФРС
    # 2% -> отлично, 5.5% -> плохо
    fed_score = 100.0 - ((fed_rate - 2.0) / (5.5 - 2.0)) * 100.0
    fed_score = clamp(fed_score)

    # Спред: если 3M почти равен ставке, это "нормальная" жёсткая денежная среда
    # если заметно ниже ставки, можно трактовать как небольшое смягчение на коротком конце
    spread = fed_rate - treasury_3m_yield

    # Небольшой бонус за то, что 3M торгуется ниже ставки
    # при спреде 0.0 бонус 0
    # при спреде 0.25 бонус ~5
    spread_bonus = clamp((spread / 0.25) * 5.0, 0.0, 8.0)

    # Смешиваем
    macro_score = 0.55 * yield_score + 0.45 * fed_score + spread_bonus
    return clamp(macro_score)


def classify_regime(score: float) -> str:
    if score >= 80:
        return "Aggressive Buy / Strong Risk-On"
    elif score >= 60:
        return "Accumulate / Mild Risk-On"
    elif score >= 40:
        return "Neutral / Wait"
    else:
        return "Defensive / Risk-Off"


def compute_market_score(inputs: MarketInputs) -> MarketResult:
    mvrv = compute_mvrv(inputs.price, inputs.realized_price)
    discount_pct = compute_discount_to_realized(inputs.price, inputs.realized_price)

    mvrv_score = score_mvrv(mvrv)
    macro_score = score_macro(inputs.treasury_3m_yield, inputs.fed_rate)

    final_score = (
        0.35 * mvrv_score
        + 0.25 * clamp(inputs.stable_score)
        + 0.20 * clamp(inputs.dominance_score)
        + 0.20 * macro_score
    )

    regime = classify_regime(final_score)

    return MarketResult(
        mvrv=round(mvrv, 6),
        discount_to_realized_pct=round(discount_pct, 2),
        mvrv_score=round(mvrv_score, 2),
        macro_score=round(macro_score, 2),
        stable_score=round(clamp(inputs.stable_score), 2),
        dominance_score=round(clamp(inputs.dominance_score), 2),
        final_score=round(final_score, 2),
        regime=regime,
    )


if __name__ == "__main__":
    # Пример на базе твоих чисел
    inputs = MarketInputs(
        price=84.05,
        realized_price=129.14581895,
        stable_score=80,          # много ликвидности в стейблах
        dominance_score=55,       # смешанная картина потоков
        treasury_3m_yield=3.678,  # ты дал это значение
        fed_rate=3.75             # ты дал это значение
    )

    result = compute_market_score(inputs)

    print("=== INPUTS ===")
    for k, v in asdict(inputs).items():
        print(f"{k}: {v}")

    print("\n=== RESULT ===")
    for k, v in asdict(result).items():
        print(f"{k}: {v}")

=== INPUTS ===
price: 84.05
realized_price: 129.14581895
stable_score: 80
dominance_score: 55
treasury_3m_yield: 3.678
fed_rate: 3.75

=== RESULT ===
mvrv: 0.650815
discount_to_realized_pct: -34.92
mvrv_score: 93.09
macro_score: 52.57
stable_score: 80
dominance_score: 55
final_score: 74.1
regime: Accumulate / Mild Risk-On


In [8]:
import requests
from datetime import datetime, timezone
from pprint import pprint


DEFI_LLAMA_STABLES_URL = "https://stablecoins.llama.fi/stablecoincharts/all"
COINGECKO_GLOBAL_URL = "https://api.coingecko.com/api/v3/global"


def fetch_json(url: str):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


def get_total_crypto_market_cap_usd() -> float:
    data = fetch_json(COINGECKO_GLOBAL_URL)

    if "data" not in data or "total_market_cap" not in data["data"]:
        raise RuntimeError(f"Unexpected CoinGecko response: {data}")

    return float(data["data"]["total_market_cap"]["usd"])


def get_stablecoin_series():
    data = fetch_json(DEFI_LLAMA_STABLES_URL)

    if not isinstance(data, list) or not data:
        raise RuntimeError(f"Unexpected DeFiLlama response: {data}")

    return data


def pick_latest_and_30d_ago(series):
    if len(series) < 31:
        raise RuntimeError("Not enough history to compute 30d change")

    latest = series[-1]
    prev_30d = series[-31]
    return latest, prev_30d


def extract_numeric_value(value):
    """Преобразует число или вложенный dict в float."""
    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, str):
        return float(value)

    if isinstance(value, dict):
        # самые вероятные ключи
        for key in [
            "peggedUSD",
            "usd",
            "value",
            "total",
            "circulating",
            "totalCirculatingUSD",
        ]:
            if key in value and isinstance(value[key], (int, float, str)):
                return float(value[key])

        # если ключи неизвестны — пробуем первое числовое значение
        for v in value.values():
            if isinstance(v, (int, float, str)):
                try:
                    return float(v)
                except ValueError:
                    pass

    raise RuntimeError(f"Cannot extract numeric value from: {value}")


def extract_total_stablecoin_market_cap(entry: dict) -> float:
    """
    Пытаемся достать total stablecoin cap из записи DeFiLlama,
    даже если формат слегка отличается.
    """
    possible_keys = [
        "totalCirculatingUSD",
        "totalCirculating",
        "circulatingUSD",
        "circulating",
        "total",
        "value",
    ]

    for key in possible_keys:
        if key in entry:
            return extract_numeric_value(entry[key])

    raise RuntimeError(f"Unexpected stablecoin entry format: {entry}")


def format_ts(ts):
    if ts is None:
        return "n/a"

    try:
        ts = float(ts)
        if ts > 10_000_000_000:
            ts /= 1000.0
        return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()
    except Exception:
        return str(ts)


def main():
    total_crypto_market_cap = get_total_crypto_market_cap_usd()
    stable_series = get_stablecoin_series()

    latest, prev_30d = pick_latest_and_30d_ago(stable_series)

    print("=== Latest entry sample ===")
    pprint(latest)

    total_stablecoin_market_cap = extract_total_stablecoin_market_cap(latest)
    total_stablecoin_market_cap_30d_ago = extract_total_stablecoin_market_cap(prev_30d)

    stablecoin_30d_change = (
        (total_stablecoin_market_cap - total_stablecoin_market_cap_30d_ago)
        / total_stablecoin_market_cap_30d_ago
        * 100.0
    )

    stablecoin_share = total_stablecoin_market_cap / total_crypto_market_cap * 100.0

    result = {
        "stablecoin_market_cap_usd": round(total_stablecoin_market_cap, 2),
        "stablecoin_market_cap_30d_ago_usd": round(total_stablecoin_market_cap_30d_ago, 2),
        "stablecoin_30d_change_pct": round(stablecoin_30d_change, 2),
        "total_crypto_market_cap_usd": round(total_crypto_market_cap, 2),
        "stablecoin_share_pct": round(stablecoin_share, 2),
        "stablecoins_latest_timestamp": format_ts(latest.get("date")),
        "stablecoins_30d_ago_timestamp": format_ts(prev_30d.get("date")),
    }

    print("\n=== Stablecoin Liquidity Metrics ===")
    for k, v in result.items():
        print(f"{k}: {v}")


if __name__ == "__main__":
    main()

=== Latest entry sample ===
{'date': '1775692800',
 'totalBridgedToUSD': {'peggedARS': 0,
                       'peggedAUD': 0,
                       'peggedCAD': 0,
                       'peggedCHF': 0,
                       'peggedCNY': 0,
                       'peggedCOP': 0,
                       'peggedEUR': 0,
                       'peggedGBP': 0,
                       'peggedGHS': 0,
                       'peggedJPY': 0,
                       'peggedKES': 0,
                       'peggedMXN': 0,
                       'peggedNGN': 0,
                       'peggedPHP': 0,
                       'peggedREAL': 0,
                       'peggedRUB': 0,
                       'peggedSGD': 0,
                       'peggedTRY': 0,
                       'peggedUAH': 0,
                       'peggedUSD': 0,
                       'peggedVAR': 0,
                       'peggedXOF': 0,
                       'peggedZAR': 0},
 'totalCirculating': {'peggedARS': 10898765.87,
  

In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class MarketInputs:
    price: float
    realized_price: float
    btc_dominance: float
    sol_dominance: float
    stablecoin_share: float
    stablecoin_30d_change: float
    treasury_3m_yield: float
    fed_rate: float

def clamp(x, lo=0.0, hi=100.0):
    return max(lo, min(hi, x))

def compute_mvrv(price: float, realized_price: float) -> float:
    if realized_price <= 0:
        raise ValueError("realized_price must be > 0")
    return price / realized_price

def score_mvrv(mvrv: float) -> float:
    if mvrv <= 0.60:
        return 95.0
    elif mvrv <= 1.00:
        return 95.0 - (mvrv - 0.60) / 0.40 * 15.0
    elif mvrv <= 1.50:
        return 80.0 - (mvrv - 1.00) / 0.50 * 30.0
    elif mvrv <= 2.00:
        return 50.0 - (mvrv - 1.50) / 0.50 * 30.0
    elif mvrv <= 2.50:
        return 20.0 - (mvrv - 2.00) / 0.50 * 15.0
    return 5.0

def score_macro(treasury_3m_yield: float, fed_rate: float) -> float:
    yield_score = 100.0 - ((treasury_3m_yield - 2.0) / (5.5 - 2.0)) * 100.0
    fed_score = 100.0 - ((fed_rate - 2.0) / (5.5 - 2.0)) * 100.0
    spread = fed_rate - treasury_3m_yield
    spread_bonus = clamp((spread / 0.25) * 5.0, 0.0, 8.0)
    return clamp(0.55 * clamp(yield_score) + 0.45 * clamp(fed_score) + spread_bonus)

def calc_stable_score(stablecoin_share: float, stablecoin_30d_change: float) -> float:
    base = 50 + (stablecoin_share - 10.0) * 8
    trend = stablecoin_30d_change * 10
    return clamp(base + trend)

def calc_dominance_score(sol_dominance: float, btc_dominance: float, stablecoin_share: float) -> float:
    base = 50
    sol_part = (sol_dominance - 1.5) * 20
    btc_part = -(btc_dominance - 55.0) * 3
    stable_part = -(stablecoin_share - 10.0) * 2
    return clamp(base + sol_part + btc_part + stable_part)

def classify(score: float) -> str:
    if score >= 80:
        return "Aggressive Buy / Strong Risk-On"
    elif score >= 60:
        return "Accumulate / Mild Risk-On"
    elif score >= 40:
        return "Neutral / Wait"
    return "Defensive / Risk-Off"

def compute_market_score(inputs: MarketInputs):
    mvrv = compute_mvrv(inputs.price, inputs.realized_price)
    mvrv_score = score_mvrv(mvrv)
    stable_score = calc_stable_score(inputs.stablecoin_share, inputs.stablecoin_30d_change)
    dominance_score = calc_dominance_score(inputs.sol_dominance, inputs.btc_dominance, inputs.stablecoin_share)
    macro_score = score_macro(inputs.treasury_3m_yield, inputs.fed_rate)

    final_score = (
        0.35 * mvrv_score
        + 0.25 * stable_score
        + 0.20 * dominance_score
        + 0.20 * macro_score
    )

    return {
        "mvrv": round(mvrv, 6),
        "mvrv_score": round(mvrv_score, 2),
        "stable_score": round(stable_score, 2),
        "dominance_score": round(dominance_score, 2),
        "macro_score": round(macro_score, 2),
        "final_score": round(final_score, 2),
        "regime": classify(final_score),
    }

inputs = MarketInputs(
    price=84.05,
    realized_price=129.14581895,
    btc_dominance=59.48,
    sol_dominance=1.993,
    stablecoin_share=12.47,
    stablecoin_30d_change=1.14,
    treasury_3m_yield=3.678,
    fed_rate=3.75,
)

print(compute_market_score(inputs))

{'mvrv': 0.650815, 'mvrv_score': 93.09, 'stable_score': 69.88, 'dominance_score': 47.48, 'macro_score': 52.57, 'final_score': 70.06, 'regime': 'Accumulate / Mild Risk-On'}
